# Model 2 — Strategy Personalizer (SYNTHETIC)

Component 1 · Anchor AI (J26-IT-366). All results here are **SYNTHETIC**. Model 2 learns which replanning strategy each
student prefers (a pooled LinUCB bandit). This notebook checks that the 5 strategies really give different plans, then
simulates students deciding on the plans they are shown and plots how each policy's acceptance changes over time.
**The simulated accept probabilities (0.85 when the shown strategy is the persona's preferred one, 0.3 otherwise) are
assumptions, not measured values.**

**Cell 1 — setup (run first).** In Google Colab this clones the C1 branch `dev/ravindu` of the team repo into
`/content/AnchorAI` **only if it is not there yet** (running it again never makes a nested copy), goes to that folder,
and installs only what these notebooks need (`components/c1_disruption/notebooks/requirements.txt`: numpy, scipy,
scikit-learn, quantile-forest, joblib, pandas, matplotlib — not the web server). It prints the versions. If a package
that was already loaded had to be replaced (usually numpy), it stops with a clear message: then use
**Runtime → Restart session** and run again from **cell 2** (the next code cell). Outside Colab it uses the repository
the notebook is in and installs nothing. C1 supports Python 3.11–3.13.

In [ ]:
import importlib.metadata as md, os, subprocess, sys

REPO = "https://github.com/DulanjaR/AnchorAI.git"
IN_COLAB = "google.colab" in sys.modules
PACKAGES = {"numpy": "numpy", "scipy": "scipy", "scikit-learn": "sklearn", "quantile-forest": "quantile_forest",
            "joblib": "joblib", "pandas": "pandas", "matplotlib": "matplotlib"}


def repo_root():
    if IN_COLAB:
        return "/content/AnchorAI"
    here = os.path.abspath(os.getcwd())
    while not os.path.isdir(os.path.join(here, "components", "c1_disruption")):
        if os.path.dirname(here) == here:
            raise FileNotFoundError("Open this notebook inside the AnchorAI repository (or in Colab).")
        here = os.path.dirname(here)
    return here


def installed():
    versions = {}
    for package in PACKAGES:
        try:
            versions[package] = md.version(package)
        except md.PackageNotFoundError:
            versions[package] = None
    return versions


ROOT = repo_root()
if IN_COLAB and not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "--depth", "1", "-b", "dev/ravindu", REPO, ROOT], check=True)
elif IN_COLAB:
    print("Already cloned at", ROOT, "- not cloning again.")
os.chdir(ROOT)
before = installed()
loaded = {p: getattr(sys.modules[m], "__version__", None) for p, m in PACKAGES.items() if m in sys.modules}
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    "components/c1_disruption/notebooks/requirements.txt"], check=True)
after = installed()
print("Python", sys.version.split()[0], "| folder", ROOT)
for package, version in after.items():
    print(f"  {package:16} {version}" + ("" if before[package] == version else f"   (was {before[package]})"))
replaced = [p for p, v in loaded.items() if v is not None and v != after[p]]
if replaced:
    print("\n" + "!" * 72 + f"\n{', '.join(replaced)} changed while already loaded.\n"
          "Runtime → Restart session, then run from cell 2 (the next code cell).\n" + "!" * 72)
    raise SystemExit("Restart needed: Runtime → Restart session, then run from cell 2.")

**Cell 2 — start here after a restart.** Goes to the repository folder, makes C1 importable, prints the versions in
use and starts the progress clock (every long step prints a line like `[  12.4s] training fold 3/5…`).

In [ ]:
import os, sys, tempfile

IN_COLAB = "google.colab" in sys.modules
ROOT = "/content/AnchorAI"
if not IN_COLAB:
    ROOT = os.path.abspath(os.getcwd())
    while not os.path.isdir(os.path.join(ROOT, "components", "c1_disruption")):
        ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
# Files this notebook writes go here, never into the repository.
OUT = "/content/c1_outputs" if IN_COLAB else os.path.join(tempfile.gettempdir(), "c1_outputs")
os.makedirs(OUT, exist_ok=True)

import joblib, matplotlib, numpy, pandas, quantile_forest, scipy, sklearn
print("Python", sys.version.split()[0], "| numpy", numpy.__version__, "| scipy", scipy.__version__,
      "| scikit-learn", sklearn.__version__, "| quantile-forest", quantile_forest.__version__,
      "| joblib", joblib.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Outputs go to", OUT)
from components.c1_disruption.ml.progress import Progress
progress = Progress()

**Which data.** The SYNTHETIC persona files in `components/c1_disruption/sim/data/`: personas, their special habits,
30 days of planned tasks, each day's schedule and each persona's disruption profile (format:
`components/c1_disruption/sim/README.md`). Everything produced from them is **SYNTHETIC**: it checks that the pipeline
works; it is not a research finding.

In [ ]:
DATA = "components/c1_disruption/sim/data/"
PERSONAS, HABITS = DATA + "SYNTHETIC_personas.csv", DATA + "SYNTHETIC_habits.csv"
TASKS, SCHEDULE = DATA + "SYNTHETIC_tasks_planned.csv", DATA + "SYNTHETIC_schedule.csv"
PROFILE = DATA + "SYNTHETIC_disruption_profile.csv"
from components.c1_disruption.sim.personas import load_personas, load_schedule, load_tasks
from components.c1_disruption.sim.profile import load_profile
progress("loading the SYNTHETIC persona files…")
personas = load_personas(PERSONAS, HABITS)
tasks = load_tasks(TASKS, personas)
schedule = load_schedule(SCHEDULE, personas)
profile = load_profile(PROFILE, personas)
print(len(personas), "personas,", len(tasks), "planned tasks,", len(schedule), "scheduled days")

**Disruption days and the strategy catalogue.** The disruption days are replanned (raw and Model-1-corrected plans,
Model 1 retrained weekly — the slow part). Then the distinctness check: how often two strategies give a different
plan on the same day. A pair that never differs would be pointless for Model 2 to choose between.

In [ ]:
from components.c1_disruption.ml.bandit.actions import ACTIONS, distinctness
from components.c1_disruption.sim.events import build_events
from components.c1_disruption.sim.generate import generate
progress("generating the SYNTHETIC world (seed 11)…")
world = generate(personas, tasks, seed=11, schedule=schedule, profile=profile)
progress("replanning the disruption days (Model 1 retrained each week)…")
events, skipped = build_events(world, n_estimators=50)
progress(f"{len(events)} events; checking how often the {len(ACTIONS)} strategies differ…")
report = distinctness([e.engine_input for e in events])
print("strategies:", ACTIONS)
for (a, b), share in report["pairs"].items():
    print(f"  {a:15} vs {b:15} differ in {share:6.1%}")
print("never differ:", report["neverDiffer"] or "none")
print(sum(s.kind == "no_effect" for s in skipped), "disruptions had no effect (they overlap no planned task)")
print(sum(s.kind == "cannot_plan" for s in skipped), "events the engine cannot plan (e.g. a None EF value)")

**Simulated decisions (SYNTHETIC).** Each policy shows one strategy per event; the simulated persona accepts with the
assumed probabilities above. *fixed order* always shows the first strategy, *random* a random one, *most accepted*
the one this persona accepted most, *bandit* = Model 2 (pooled LinUCB with per-student parts).

In [ ]:
from statistics import mean
from components.c1_disruption.sim.experiment import model2
progress("simulating decisions for 4 policies…")
curves = model2(world, events, seed=11)
for name, values in curves.items():
    half = len(values) // 2
    print(f"{name:14} mean acceptance {mean(values):.2f} | first half {mean(values[:half]):.2f} | "
          f"second half {mean(values[half:]):.2f} | {len(values)} events")

In [ ]:
from components.c1_disruption.ml import charts
charts.acceptance_over_time(curves, "Model 2: simulated acceptance over time per policy (SYNTHETIC; accept rates assumed)");